# Benchmark — Google Colab

**Yêu cầu một lần (tạo zip và upload lên Drive):**

1. Zip toàn bộ folder `code/` (chứa `benchmark/`, `embedding_generator/`, `profile_generator/`):
```powershell
# PowerShell — chạy từ thư mục cha chứa code/  (ví dụ: D:\BKAI\)
Compress-Archive -Path code -DestinationPath code.zip
```
```bash
# Mac / Linux
zip -r code.zip code/
```

2. Upload `code.zip` lên Drive:
```
MyDrive/
  Colab Notebooks/
    code.zip    ← upload vào đây
```

**Mỗi lần chạy:**  
Colab tự giải nén `code.zip` — chỉ cần chạy notebook từ trên xuống.  
Khi cập nhật code/data: zip lại → upload đè → Colab tự giải nén lại.


## 0. Cấu hình

In [ ]:
# ── Drive ────────────────────────────────────────────────────────────────────────────
# File zip chứa toàn bộ code/ (benchmark/ + embedding_generator/ + profile_generator/)
DRIVE_CODE_ZIP   = '/content/drive/MyDrive/Colab Notebooks/code.zip'

# Thư mục lưu checkpoint & results trên Drive (tự động tạo)
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/Colab Notebooks/benchmark_output'

# ── Colab ────────────────────────────────────────────────────────────────────────────
COLAB_CODE_DIR = '/content/code'

print('Config OK')


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Giải nén code từ Drive

Nếu `code.zip` trên Drive **mới hơn** lần giải nén trước, Colab sẽ tự giải nén lại.  
Cập nhật code/data: zip lại → upload đè → chạy lại cell này.

In [ ]:
import os, zipfile, shutil

if not os.path.exists(DRIVE_CODE_ZIP):
    raise FileNotFoundError(
        f'Không tìm thấy {DRIVE_CODE_ZIP}
'
        'Hãy zip folder code/ thành code.zip và upload lên Drive.
'
        'Xem hướng dẫn trong cell header.'
    )

# Kiểm tra nếu cần giải nén lại (zip mới hơn lần trước)
zip_mtime = os.path.getmtime(DRIVE_CODE_ZIP)
flag_file  = f'{COLAB_CODE_DIR}/.extracted_mtime'

need_extract = True
if os.path.exists(flag_file):
    with open(flag_file) as f:
        prev_mtime = float(f.read().strip())
    if prev_mtime >= zip_mtime:
        need_extract = False
        print('[OK] Code đã giải nén và còn mới — bỏ qua.')

if need_extract:
    if os.path.exists(COLAB_CODE_DIR):
        shutil.rmtree(COLAB_CODE_DIR)
    print(f'Đang giải nén {DRIVE_CODE_ZIP} ...')
    with zipfile.ZipFile(DRIVE_CODE_ZIP, 'r') as z:
        z.extractall('/content')
    with open(flag_file, 'w') as f:
        f.write(str(zip_mtime))
    print(f'[OK] Giải nén xong → {COLAB_CODE_DIR}')

## 3. Link checkpoint & results lên Drive

Checkpoints và results được ghi trực tiếp lên Drive — không mất khi Colab hết session.

In [ ]:
from pathlib import Path

CODE_ROOT = Path(COLAB_CODE_DIR)
print('=== Kiểm tra cấu trúc thư mục sau giải nén ===')
for d in ['benchmark', 'embedding_generator', 'profile_generator']:
    status = 'OK' if (CODE_ROOT / d).exists() else 'THIẾU'
    print(f'  [{status}] {d}/')


## 4. Link checkpoint & results thẳng lên Drive

Mọi file `.pt` được ghi trực tiếp lên Drive trong lúc training.

In [ ]:
import os, shutil

RLMREC_DIR = f'{BENCHMARK_DIR}/external/RLMRec'

OUTPUT_LINKS = [
    (f'{BENCHMARK_DIR}/checkpoints',     f'{DRIVE_OUTPUT_DIR}/checkpoints'),
    (f'{BENCHMARK_DIR}/results',         f'{DRIVE_OUTPUT_DIR}/results'),
    (f'{RLMREC_DIR}/encoder/checkpoint', f'{DRIVE_OUTPUT_DIR}/rlmrec_checkpoint'),
    (f'{RLMREC_DIR}/encoder/log',        f'{DRIVE_OUTPUT_DIR}/rlmrec_log'),
]

for link_path, target_path in OUTPUT_LINKS:
    os.makedirs(target_path, exist_ok=True)
    if os.path.exists(link_path) and not os.path.islink(link_path):
        shutil.rmtree(link_path)
    if not os.path.islink(link_path):
        os.symlink(target_path, link_path)
        print(f'[LINK] {link_path}\n       → {target_path}')
    else:
        print(f'[OK]   {link_path}')

## 5. Cài đặt dependencies

In [ ]:
%pip install -q torch numpy pandas scipy scikit-learn sentence-transformers tqdm pyyaml

## 6. Kiểm tra GPU & môi trường

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('Không có GPU — vào Runtime > Change runtime type > GPU để bật')

## 7. Kiểm tra dữ liệu & embeddings

In [ ]:
from pathlib import Path

CODE_ROOT     = Path(COLAB_CODE_DIR)
BENCHMARK_DIR = CODE_ROOT / 'benchmark'

ml20m_dir = CODE_ROOT / 'profile_generator' / 'llm-movie-profiler-v1-20260402' / 'data' / 'ml-20m'
print('=== Dữ liệu ML-20M ===')
for f in ['ratings.csv', 'movies.csv', 'genome-scores.csv', 'genome-tags.csv']:
    status = 'OK' if (ml20m_dir / f).exists() else 'THIẾU'
    print(f'  [{status}] {f}')

emb_dir = CODE_ROOT / 'embedding_generator' / 'output' / 'bge-large-v1.5'
print('\n=== Embeddings ===')
for f in ['profile_embeddings.npy', 'mood_vectors.npy', 'theme_matrix.npy',
          'genome_embeddings.npy', 'movie_id_index.json']:
    status = 'OK' if (emb_dir / f).exists() else 'THIẾU'
    print(f'  [{status}] {f}')

print('\n=== Drive output links ===')
for name, path in [('checkpoints', BENCHMARK_DIR / 'checkpoints'),
                   ('results',      BENCHMARK_DIR / 'results')]:
    if path.is_symlink():
        print(f'  [LINK] {name}/ → {path.resolve()}')
    else:
        print(f'  [WARN] {name}/ chưa được link — chạy lại cell 4')

## 8. Setup Python path

In [ ]:
import sys
sys.path.insert(0, str(BENCHMARK_DIR))

from config import EMBED_DIM, NUM_EPOCHS, PATIENCE, FEATURE_CONFIGS
print(f'EMBED_DIM={EMBED_DIM}, NUM_EPOCHS={NUM_EPOCHS}, PATIENCE={PATIENCE}')
print(f'Feature configs: {list(FEATURE_CONFIGS.keys())}')

## 9. Chạy một thí nghiệm đơn

**CF models (LightGCN, BPR-MF, ...):**

| Tham số | Các lựa chọn |
|---|---|
| `MODEL` | `bpr_mf`, `lightgcn`, `lightgcn_sf`, `xsimgcl`, `simgcl`, `lightgcl`, `kar` |
| `FEATURES` | `none`, `genome`, `bert_title`, `llm_profile`, `llm_mood`, `llm_themes`, `llm_prof_mood`, `llm_all`, `genome_llm` |

**Sequential models — Direction 4 (SASRec):**

| Tham số | Các lựa chọn |
|---|---|
| `MODEL` | `sasrec` |
| `FEATURES` | `none` (baseline) hoặc `llm_prof_mood` (M7) |
| `INJECTION` | `none`, `input`, `ffn`, `output`, `input+ffn`, `input+output`, `ffn+output`, `all` |
| `SEQ_BATCH_SIZE` | T4 → `1024`, A100 → `2048` |

In [ ]:
import os
os.chdir(str(BENCHMARK_DIR))

# ── Tham số ──────────────────────────────────────────────────────────────
MODEL          = 'sasrec'         # đổi model ở đây
FEATURES       = 'llm_prof_mood'  # 'none' = baseline, 'llm_prof_mood' = M7
INJECTION      = 'input'          # chỉ dùng cho sasrec; bỏ qua với CF models
SEED           = 42
EPOCHS         = 100
SEQ_BATCH_SIZE = 1024             # T4=1024, A100=2048; chỉ dùng cho sasrec

# ── Chạy ─────────────────────────────────────────────────────────────────
if MODEL == 'sasrec':
    !python run_experiment.py \
        --model {MODEL} \
        --features {FEATURES} \
        --injection {INJECTION} \
        --seed {SEED} \
        --epochs {EPOCHS} \
        --seq-batch-size {SEQ_BATCH_SIZE} \
        --device auto
else:
    !python run_experiment.py \
        --model {MODEL} \
        --features {FEATURES} \
        --seed {SEED} \
        --epochs {EPOCHS} \
        --device auto

## 10. Chạy Quick Test (tất cả model, 1 seed, 20 epochs)

In [ ]:
import os
os.chdir(str(BENCHMARK_DIR))
!python run_ablation.py --quick

## 11. Chạy R1 — RLMRec-plus

In [ ]:
import os
os.chdir(f'{COLAB_CODE_DIR}/benchmark/external/RLMRec')

SEED   = 42
EPOCHS = 100

!python encoder/train_encoder.py \
    --model lightgcn_plus \
    --dataset ml20m \
    --device cuda \
    --seed {SEED} \
    --epochs {EPOCHS}

## 12. Chạy R2 — RLMRec-gene

In [ ]:
import os
os.chdir(f'{COLAB_CODE_DIR}/benchmark/external/RLMRec')

SEED   = 42
EPOCHS = 100

!python encoder/train_encoder.py \
    --model lightgcn_gene \
    --dataset ml20m \
    --device cuda \
    --seed {SEED} \
    --epochs {EPOCHS}

## 13. Kiểm tra checkpoint đã lưu trên Drive

In [ ]:
import os

ckpt_dir = f'{DRIVE_OUTPUT_DIR}/checkpoints'
if os.path.exists(ckpt_dir):
    experiments = sorted(os.listdir(ckpt_dir))
    print(f'Checkpoint trên Drive ({len(experiments)} experiments):')
    for exp in experiments:
        files = os.listdir(f'{ckpt_dir}/{exp}')
        print(f'  {exp}/  {files}')
else:
    print('Chưa có checkpoint nào.')